# MLflow Example: Tracking Two Model Training Runs

In [11]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [12]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [13]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [14]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [15]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [16]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='ChenAuCarre', repo_name='mlflow-with-dagshub', mlflow=True)

Initialized MLflow to track repo "ChenAuCarre/mlflow-with-dagshub"

Repository ChenAuCarre/mlflow-with-dagshub initialized!

In [17]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [18]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/d76e89807ce34ea08e348d5b7b15647f', creation_time=1794844716342, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794844716342, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [19]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        # MLflow 3 serializes with skops, which rejects tree types unless trusted
        mlflow.sklearn.log_model(model, run_name,
                                  skops_trusted_types=["sklearn.tree._tree.Tree"])

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [ ]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

In [ ]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1_0_2")

In [ ]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")